# Evaluation

Every number quoted in the README's **Results** section is computed here. The point
of this notebook is that no claim in the documentation should rest on a script that
does not ship with the repository.

Run after `train_model.ipynb` and `shape_optimization.ipynb`.

In [1]:
import sys, json
sys.path.insert(0, "..")

import numpy as np
import torch

from src.data import read_split, make_val_split, load_mesh
from src.dataset import build_all, faces_to_edge_index
from src.model import MeshSurrogate
from src import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"

with open("../outputs/norm_stats.json") as f:
    stats = json.load(f)

train_ids, val_ids = make_val_split(read_split("train.txt"), n_val=50, seed=0)
train_samples = build_all(train_ids, "train")
test_samples = build_all(read_split("test.txt"), "test")
test_pyg = torch.load("../outputs/cache/test_pyg.pt", weights_only=False)

model = MeshSurrogate(in_channels=6).to(device)
model.load_state_dict(torch.load("../outputs/checkpoints/best_model.pt", weights_only=True, map_location=device))
model.eval()

pressure_pred, cd_pred = evaluate.predict_test_set(model, test_pyg, stats, device)
print(f"{len(train_samples)} train / {len(val_ids)} val / {len(test_samples)} test")

450 train / 50 val / 111 test


## Pressure

The model is scored against two predictors that use no information about the specific
car. The second one matters: since every car in the dataset is broadly car-shaped
(high pressure at the nose, low over the roof), reciting the training set's average
pressure field is already a decent predictor. A model that failed to clearly beat it
would have learned nothing about *this* car's geometry while still posting a
respectable-looking error.

In [2]:
P_train = np.stack([s["pressure"] for s in train_samples])
P_test = np.stack([s["pressure"] for s in test_samples])

baselines = evaluate.pressure_baselines(P_train, P_test)
model_mse = float(((P_test - pressure_pred) ** 2).mean())

print(f"{'predictor':<45}{'MSE':>10}{'RMSE':>10}")
print("-" * 65)
for name, mse in [("global scalar mean", baselines["global_mean"]),
                  ("per-vertex mean field (geometry-blind)", baselines["template_field"]),
                  ("trained model", model_mse)]:
    print(f"{name:<45}{mse:>10.1f}{np.sqrt(mse):>10.2f}")

beyond = 100 * (1 - model_mse / baselines["template_field"])
print(f"\nvariance explained beyond the template: {beyond:.1f}%")
print(f"per-vertex pressure spread (std): {P_train.std():.1f}")

predictor                                           MSE      RMSE
-----------------------------------------------------------------
global scalar mean                               2379.6     48.78
per-vertex mean field (geometry-blind)            250.7     15.83
trained model                                      72.5      8.52

variance explained beyond the template: 71.1%
per-vertex pressure spread (std): 48.0


## Drag coefficient

`RMSE` is the number that matters downstream: it is in the target's own units, so a
shape-optimization run claiming an improvement smaller than this RMSE has claimed
something indistinguishable from noise.

The analytic route re-derives Cd from the model's *predicted pressure* via the same
formula used to build the labels. It is reported here because the README treats it as
a consistency check -- and because measuring it shows it is the **less** accurate of
the two estimators, which is why it is not treated as independent verification.

In [3]:
cd_true = np.array([s["cd"] for s in test_samples])
cd_analytic = evaluate.analytic_cd_from_pressure(test_samples, pressure_pred)
cd_train_mean = np.array([s["cd"] for s in train_samples]).mean()

rows = [
    ("predict training mean", np.full_like(cd_true, cd_train_mean)),
    ("Cd head", cd_pred),
    ("analytic, from predicted pressure", cd_analytic),
]

print(f"{'predictor':<40}{'MSE':>9}{'RMSE':>8}{'R2':>8}")
print("-" * 65)
metrics = {}
for name, pred in rows:
    m = evaluate.regression_metrics(pred, cd_true)
    metrics[name] = m
    print(f"{name:<40}{m['mse']:>9.2f}{m['rmse']:>8.2f}{m['r2']:>8.3f}")

cd_rmse = metrics["Cd head"]["rmse"]
print(f"\nCd spread across test cars (std): {cd_true.std():.2f}")
print(f"typical |Cd|: {np.abs(cd_true).mean():.2f}")

predictor                                     MSE    RMSE      R2
-----------------------------------------------------------------
predict training mean                       42.38    6.51  -0.020
Cd head                                      5.43    2.33   0.869
analytic, from predicted pressure           15.39    3.92   0.630

Cd spread across test cars (std): 6.45
typical |Cd|: 59.04


## The reference area

Cd divides drag by a reference area, so that area is part of the optimization target.
An earlier version used the bounding-box cross-section; this uses the true projected
(silhouette) area. The gap between them is measured here: a bounding box is set by
four extreme vertices, is barely differentiable, and over-estimates the area a car
actually presents to the flow.

In [4]:
from src.data import projected_frontal_area

projected = np.array([projected_frontal_area(s["pos"], s["faces"]) for s in train_samples])
extent = np.array([s["pos"].max(0) - s["pos"].min(0) for s in train_samples])
bbox = extent[:, 0] * extent[:, 1]

print(f"{'reference area definition':<40}{'mean':>10}{'std':>10}")
print("-" * 60)
print(f"{'bounding-box cross-section':<40}{bbox.mean():>10.4f}{bbox.std():>10.4f}")
print(f"{'projected (silhouette) area':<40}{projected.mean():>10.4f}{projected.std():>10.4f}")
print(f"\nbounding box over-estimates by {100 * (bbox.mean() / projected.mean() - 1):.1f}% on average")
print(f"projected / bbox ratio: mean {(projected / bbox).mean():.3f}, "
      f"spread {(projected / bbox).std():.3f} (so the two are not a fixed rescaling)")

area_stats = {"bbox_mean": float(bbox.mean()), "projected_mean": float(projected.mean()),
              "bbox_overestimate_pct": float(100 * (bbox.mean() / projected.mean() - 1))}

reference area definition                     mean       std
------------------------------------------------------------
bounding-box cross-section                  0.4530    0.0763
projected (silhouette) area                 0.4223    0.0726

bounding box over-estimates by 7.3% on average
projected / bbox ratio: mean 0.932, spread 0.028 (so the two are not a fixed rescaling)


## Shape optimization

Two questions, neither answerable from the optimizer's own reported improvement.

**Is the deformation a coherent reshaping, or surface noise?** Displacement magnitude
alone cannot tell these apart. `deformation_roughness` measures the fraction of the
displacement field that is high-frequency: ~0 is smooth, ~1 is per-vertex noise. The
reference values below calibrate the scale.

**Is the claimed improvement bigger than the model's own error bar?** If the change is
smaller than the Cd head's test RMSE, it is indistinguishable from the model simply
being imprecise.

In [5]:
result = torch.load("../outputs/shape_opt_result.pt", weights_only=False)
faces = result["faces"]
edge_index = faces_to_edge_index(faces)
pos_before = np.asarray(result["pos_history"][0])
pos_after = np.asarray(result["pos_history"][-1])
displacement = pos_after - pos_before

rough = evaluate.deformation_roughness(displacement, edge_index)

rng = np.random.default_rng(0)
smooth_mode = np.zeros_like(pos_before)
smooth_mode[:, 1] = 0.01 * np.sin(3 * pos_before[:, 2])

print(f"car {result['sample_id']}")
print(f"  max vertex displacement : {np.linalg.norm(displacement, axis=1).max():.4f}  (mesh spans ~2.0)")
print(f"  ROUGHNESS               : {rough:.3f}")
print()
print("  calibration:")
print(f"    smooth low-order mode : {evaluate.deformation_roughness(smooth_mode, edge_index):.3f}")
print(f"    pure random noise     : {evaluate.deformation_roughness(rng.normal(size=displacement.shape), edge_index):.3f}")

car 753
  max vertex displacement : 0.0335  (mesh spans ~2.0)
  ROUGHNESS               : 0.022

  calibration:
    smooth low-order mode : 0.009
    pure random noise     : 1.094


In [6]:
with open("../outputs/shape_opt_verification_multi.json") as f:
    multi = json.load(f)

nf = evaluate.noise_floor_ratio(multi, cd_rmse)
drag = np.array([r["drag_force_pct_change"] for r in multi])
area = np.abs([r["frontal_area_pct_change"] for r in multi])
vol = np.abs([r["volume_pct_change"] for r in multi])

print(f"across {len(multi)} held-out shapes")
print(f"  drag force reduced on   : {(drag < 0).sum()}/{len(drag)}")
print(f"  mean drag force change  : {drag.mean():+.2f}%   (range {drag.min():+.2f}% to {drag.max():+.2f}%)")
print(f"  max frontal-area drift  : {area.max():.2f}%")
print(f"  max volume drift        : {vol.max():.2f}%")
print()
print("  vs the surrogate's own noise floor:")
print(f"    mean |Cd change|      : {nf['mean_abs_cd_change']:.2f}")
print(f"    Cd head test RMSE     : {nf['cd_rmse']:.2f}")
print(f"    ratio                 : {nf['ratio']:.2f}x   ({nf['n_exceeding_rmse']}/{nf['n_total']} shapes exceed the RMSE)")

if nf["ratio"] < 1:
    print("\n  WARNING: claimed improvement is below the model's own error bar.")

across 10 held-out shapes
  drag force reduced on   : 10/10
  mean drag force change  : -10.93%   (range -14.98% to -7.64%)
  max frontal-area drift  : 0.36%
  max volume drift        : 0.42%

  vs the surrogate's own noise floor:
    mean |Cd change|      : 6.29
    Cd head test RMSE     : 2.33
    ratio                 : 2.70x   (10/10 shapes exceed the RMSE)


## Summary

Written to `outputs/evaluation_metrics.json` so the README's tables can be checked
against a file rather than transcribed by hand.

In [7]:
summary = {
    "pressure": {"global_mean_mse": baselines["global_mean"],
                 "template_field_mse": baselines["template_field"],
                 "model_mse": model_mse,
                 "pct_beyond_template": beyond},
    "cd": {name: m for name, m in metrics.items()},
    "reference_area": area_stats,
    "shape_optimization": {"sample_id": result["sample_id"],
                           "roughness": rough,
                           "max_displacement": float(np.linalg.norm(displacement, axis=1).max()),
                           "n_improved": int((drag < 0).sum()),
                           "n_shapes": len(drag),
                           "mean_drag_force_pct": float(drag.mean()),
                           "max_frontal_area_drift_pct": float(area.max()),
                           "max_volume_drift_pct": float(vol.max()),
                           **nf},
}

with open("../outputs/evaluation_metrics.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "pressure": {
    "global_mean_mse": 2379.63037109375,
    "template_field_mse": 250.6766357421875,
    "model_mse": 72.53939819335938,
    "pct_beyond_template": 71.0625611443247
  },
  "cd": {
    "predict training mean": {
      "mse": 42.376747131347656,
      "rmse": 6.509742478112914,
      "r2": -0.019909977912902832
    },
    "Cd head": {
      "mse": 5.42727192372562,
      "rmse": 2.3296506012116107,
      "r2": 0.8693781495094299
    },
    "analytic, from predicted pressure": {
      "mse": 15.390459593015471,
      "rmse": 3.923067625343141,
      "r2": 0.6295873522758484
    }
  },
  "reference_area": {
    "bbox_mean": 0.4530039429664612,
    "projected_mean": 0.4222570039828618,
    "bbox_overestimate_pct": 7.281569919168773
  },
  "shape_optimization": {
    "sample_id": "753",
    "roughness": 0.022126160178590905,
    "max_displacement": 0.0334782749414444,
    "n_improved": 10,
    "n_shapes": 10,
    "mean_drag_force_pct": -10.92577175891685,
    "max_frontal_